In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without scattered import errors.
import sys
import os

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import mne  # noqa: E402
from mne.viz import plot_topomap  # noqa: E402
from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    load_analyzers,
    analyzers_to_datasets,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

# Inverted Subject–Frequency Features ICA on Wavelet Power

**Approach 9** — observations = C×T, features = S×F.

This is the **exact inversion** of Approach 8 (`wavelet_ica_subject_freq_features.ipynb`),
which uses observations = S×F, features = C×T.

Pipeline: z-score along time → transpose to `(S, F, C, T)` →
reshape to `(C×T, S×F)` → PCA (20) → FastICA (10).

ICA discovers **subject–spectral modes** — recurring subject × frequency
patterns that appear independently across channel–time observations.
The ICA **sources** (`(C×T, K)`) are the channel–time activations of each mode,
and the ICA **mixing matrix** (`(S×F, K)`) describes which subject–frequency
combinations participate.

| Aspect | Approach 8 (Subject-Freq) | Approach 9 (Inverted, here) |
|--------|---------------------------|-----------------------------|
| Observations | S × F | C × T |
| Features | C × T | S × F |
| Components represent | Spatial-temporal modes `(C, T)` | Subject-spectral modes `(S, F)` |
| Time info lives in | Components (axis 2) | Sources (reshaped to C, T) |


## Configuration


In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Wavelet parameters ───────────────────────────────────────────────────────────────
FREQS = np.linspace(1, 40, 20)
WAVELET_CACHE = ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"

# ── Decomposition parameters ────────────────────────────────────────────────────────
N_COMPONENTS_PCA = 20
N_COMPONENTS_ICA = 10
ICA_RANDOM_STATE = 42

# ── Subset sizes (keep notebooks interactive) ───────────────────────────────────
MAX_SUBJECTS = 5
MAX_CHANNELS = 32
MAX_TIMES = 10_000

# ── Plot saving ─────────────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "inverted_subject_freq_features"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## Data Loading


In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    False,
    n_jobs=1,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)
print(f"Datasets loaded: {list(datasets.keys())}")

## Load or Compute Wavelet Transforms


In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    wavelet_dir=WAVELET_CACHE,
    reuse_wavelets=True,
    representation="power",
)
print(f"Broadband datasets: {list(broadband_datasets.keys())}")

## Dataset Selection


In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data
sfreq = bb_ad.sfreq

# Subset for interactive speed
n_subjects = min(bb_data.shape[0], MAX_SUBJECTS)
n_channels = min(bb_data.shape[1], MAX_CHANNELS)
n_freqs = bb_data.shape[2]
n_times = min(bb_data.shape[3], MAX_TIMES)
bb_data = bb_data[:n_subjects, :n_channels, :, :n_times]

time = np.arange(n_times) / sfreq

print(f"Dataset         : {LABEL}")
print(f"Shape (S,C,F,T) : {bb_data.shape}")
print(f"sfreq           : {sfreq} Hz")

---
## Step 1 — Z-score and Reshape (Inverted)

**Z-scoring** normalises each `(subject, channel, frequency)` time series
to zero mean and unit variance.

**Reshaping** transposes and reshapes so that **channel × time is the
observation axis** and **subject × frequency is the feature axis**:

```
(S, C, F, T)  →  transpose to  (S, F, C, T)
              →  reshape to     (S × F,  C × T)
              →  transpose to   (C × T,  S × F)
                                observations  features
```

Each row of the resulting 2-D matrix is a single channel–time-point
combination described by its subject × frequency loading profile.

### How this differs from subject-freq features (Approach 8)

Approach 8 uses `(S×F, C×T)` — subjects and frequencies observe
channel–time patterns.  Here we flip: `(C×T, S×F)` — channels and
time observe subject–frequency patterns.  ICA now discovers
**subject–spectral modes** rather than spatial-temporal modes.


In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape: (S, C, F, T) → transpose → (S, F, C, T) → (S*F, C*T) → transpose → (C*T, S*F)
bb_z_sf = bb_z.transpose(0, 2, 1, 3)  # (S, F, C, T)
n_obs_orig = n_subjects * n_freqs
n_feat_orig = n_channels * n_times
X_inv = bb_z_sf.reshape(n_obs_orig, n_feat_orig).T  # (C*T, S*F)

print(f"Inverted matrix shape : {X_inv.shape}")
print(f"  Observations (C×T)  : {X_inv.shape[0]}")
print(f"  Features     (S×F)  : {X_inv.shape[1]}")
print(f"Column means ≈ 0 : {X_inv.mean(axis=0).mean():.6f}")
print(f"Column stds       : {X_inv.std(axis=0).mean():.6f}")

---
## Step 2 — PCA Dimensionality Reduction + ICA Decomposition

PCA reduces the `S × F` feature space to `N_COMPONENTS_PCA` directions,
then FastICA rotates to maximise independence.

**Results:**

| Object | Shape | Description |
|--------|-------|-------------|
| `ica_sources` | `(C×T, K)` | Channel–time activation of each IC |
| `ica_full_mixing` | `(S×F, K)` | Subject–frequency mixing weights |
| `mixing_2d` | `(S, F, K)` | Mixing reshaped to subject × frequency |
| `sources_2d` | `(K, C, T)` | Sources reshaped to channel × time |

### How this differs from subject-freq features (Approach 8)

In Approach 8, ICA scores reshape to `(S, F, K)` and components to `(K, C, T)`.
Here, the ICA **mixing matrix** reshapes to `(S, F, K)` (same subject–frequency
structure) and the ICA **sources** reshape to `(K, C, T)` (same spatial-temporal
structure).  The roles of scores/components vs sources/mixing are swapped.


In [ ]:
# --- PCA ---
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca_scores = pca.fit_transform(X_inv)  # (C*T, K_pca)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title(f"PCA Scree Plot — {LABEL}")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance — {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"Top {N_COMPONENTS_PCA} components explain "
    f"{cumulative[-1] * 100:.1f}% of total variance."
)

# --- ICA ---
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
ica_sources = ica.fit_transform(pca_scores)  # (C*T, K_ica)

# Full feature-space mixing matrix: (S*F, K_ica)
# ICA mixing in PCA space: (K_pca, K_ica)
# PCA loadings: (K_pca, S*F)
# Full mixing = PCA_components^T @ ICA_mixing → (S*F, K_ica)
ica_full_mixing = pca.components_.T @ ica.mixing_  # (S*F, K_ica)

# Reshape for downstream analysis
mixing_2d = ica_full_mixing.reshape(n_subjects, n_freqs, N_COMPONENTS_ICA)  # (S, F, K)
sources_2d = ica_sources.T.reshape(
    N_COMPONENTS_ICA, n_channels, n_times
)  # (K, C, T)  [sources are (C*T, K), transpose to (K, C*T), reshape]

print(f"ICA sources shape      : {ica_sources.shape}")
print(f"Full mixing shape      : {ica_full_mixing.shape}")
print(f"Mixing 2-D shape       : {mixing_2d.shape}")
print(f"Sources 2-D shape      : {sources_2d.shape}")

---
## Analysis (a) — Intersubject Correlation Matrix of ICA Components

For each ICA component we compute a **subject × subject** Pearson
correlation matrix.  Each subject is represented by their
**frequency mixing-weight vector** (shape `F`).

High off-diagonal correlations indicate that the component has a
consistent spectral mixing profile across individuals.

### How this differs from subject-freq features (Approach 8)

In Approach 8, each subject’s frequency vector comes from ICA **scores**
`(S, F, K)`.  Here, it comes from the ICA **mixing matrix** `(S, F, K)`.
Both measure spectral consistency across subjects, but from opposite
sides of the decomposition.


In [ ]:
# Per-subject frequency mixing vector for each IC
# mixing_2d: (S, F, K)
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each subject's (F,) frequency-mixing vector for IC i
    corr_mat = np.corrcoef(mixing_2d[:, :, i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Intersubject Correlation of IC Freq Mixing Vectors — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "isc_component_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (b) — Component Temporal Profiles (Mean ± Std Across Subjects)

Each ICA source reshapes to `(C, T)`.  Channel-averaging gives the
**temporal profile** of each mode.  To assess per-subject variability,
we project each subject’s data through the mixing weights:

For subject *s* and component *k*:

$$a_{s,k}(t) = \frac{1}{F}\sum_{f} \text{mixing}_{s,f,k} \;\cdot\;
\frac{1}{C}\sum_{c} w_{k,c} \;\cdot\; z_{s,c,f}(t)$$

### How this differs from subject-freq features (Approach 8)

In Approach 8, temporal profiles come from ICA **components** `(K, C, T)`.
Here, they come from ICA **sources** reshaped to `(K, C, T)`.  The
per-subject weighting uses mixing weights instead of scores.


In [ ]:
# Channel-averaged source temporal profiles: (K, T)
source_time_profiles = sources_2d.mean(axis=1)  # (K, T)

# Per-subject temporal activations:
# Channel weights for each IC: time-averaged source pattern → (K, C)
channel_weights = sources_2d.mean(axis=2)  # (K, C)

# Per-subject projection:
# bb_z: (S, C, F, T),  channel_weights: (K, C),  mixing_2d: (S, F, K)
# Step 1: weight data by channel pattern → (S, K, F, T)
weighted_data = np.einsum("kc,scft->skft", channel_weights, bb_z)
# Step 2: weight by frequency mixing → (S, K, T)
subject_temporal = np.einsum("sfk,skft->skt", mixing_2d, weighted_data) / (
    n_freqs * n_channels
)  # (S, K, T)

mean_temporal = subject_temporal.mean(axis=0)  # (K, T)
std_temporal = subject_temporal.std(axis=0)  # (K, T)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, mean_temporal[i], lw=0.8, color="mediumpurple", label="mean")
    ax.fill_between(
        time,
        mean_temporal[i] - std_temporal[i],
        mean_temporal[i] + std_temporal[i],
        alpha=0.25,
        color="mediumpurple",
        label="± 1 std",
    )
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} — Temporal Profile", fontsize=10)
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"ICA Mode Temporal Profiles (mean ± std across subjects) — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_temporal_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (c) — Frequency × Time Mean-Loading Heatmap

For each ICA source, we compute the **mean loading per subject**
at each wavelet frequency and time point.  The IC channel weights
are derived from the source `(K, C, T)` by averaging over time,
then used to project the z-scored data:

```
channel_weights(k, c) = mean_t[ sources_2d(k, c, t) ]
weighted_data(s,k,f,t) = sum_c[ channel_weights(k,c) × bb_z(s,c,f,t) ]
loading(k, f, t) = mean_s[ mixing_2d(s,f,k) × weighted_data(s,k,f,t) ] / C
```

### How this differs from subject-freq features (Approach 8)

In Approach 8, the loading uses ICA **scores** and **component** channel
weights.  Here, it uses ICA **mixing** weights and **source** channel
weights — same frequency × time structure, opposite decomposition side.


In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
# Channel weights from sources: mean over time
channel_weights = sources_2d.mean(axis=2)  # (K, C)
# Project bb_z through channel weights
weighted_data = np.einsum("kc,scft->skft", channel_weights, bb_z)  # (S,K,F,T)
# Combine with mixing_2d: loading(k,f,t) = mean_s[ mixing(s,f,k) * weighted(s,k,f,t) ] / C
ft_loading = np.einsum("sfk,skft->kft", mixing_2d, weighted_data) / (
    n_subjects * n_channels
)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_loading[i]  # (F, T)
    vmin_s, vmax_s = np.percentile(data_i, 1), np.percentile(data_i, 99)
    ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="inferno",
        vmin=vmin_s,
        vmax=vmax_s,
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} \u2014 Freq \u00d7 Time Mean Loading", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency \u00d7 Time Mean Loading per Mode \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


---
## Analysis (d) — Mean and Variance of Source Channel Loadings as Topomaps

Channel loadings are obtained from the ICA **sources** (reshaped to
`(K, C, T)`) by averaging over time → `(K, C)`.  To assess
inter-individual variability, we combine the per-subject frequency
mixing weights with the source channel loadings to get per-subject
channel loadings `(S, C, K)`, then compute mean and variance.

### How this differs from subject-freq features (Approach 8)

In Approach 8, channel loadings come from ICA **components** `(K, C, T)`
and per-subject weighting uses **scores** `(S, F, K)`.  Here, channel
loadings come from ICA **sources** `(K, C, T)` and per-subject weighting
uses **mixing weights** `(S, F, K)`.  Same structure, opposite
decomposition side.


In [ ]:
# Source channel loadings: time-averaged → (K, C)
source_channel_loadings = sources_2d.mean(axis=2)  # (K, C)

# Per-subject channel loadings:
# Weight source channel pattern by each subject's mean frequency mixing
# mixing_2d: (S, F, K) → mean over F → (S, K)
subject_mean_mixing = mixing_2d.mean(axis=1)  # (S, K)
# Per-subject channel loading: (S, K) × (K, C) → (S, C, K)
ica_channel_loadings = np.einsum(
    "sk,kc->sck", subject_mean_mixing, source_channel_loadings
)  # (S, C, K)

# Mean and variance across subjects
ica_ch_mean = ica_channel_loadings.mean(axis=0)  # (C, K)
ica_ch_var = ica_channel_loadings.var(axis=0)  # (C, K)

# Get MNE Info for topomap
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_ICA)

# --- Mean topomaps ---
_vlim_mean = np.percentile(np.abs(ica_ch_mean[:, :n_show]), 99)
fig_mean, axes_mean = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_mean = [axes_mean]

for i, ax in enumerate(axes_mean):
    im, _ = plot_topomap(
        ica_ch_mean[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        vlim=(-_vlim_mean, _vlim_mean),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_mean.suptitle(
    f"Mean Source Channel Loading (topomap) — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_mean[-1], label="mean loading")
fig_mean.tight_layout()
if SAVE_PLOTS:
    fig_mean.savefig(PLOTS_DIR / "ica_topomap_mean.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

# --- Variance topomaps ---
_vmax_var = np.percentile(ica_ch_var[:, :n_show], 99)
fig_var, axes_var = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_var = [axes_var]

for i, ax in enumerate(axes_var):
    im, _ = plot_topomap(
        ica_ch_var[:, i],
        info,
        axes=ax,
        show=False,
        cmap="YlOrRd",
        vlim=(0, _vmax_var),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_var.suptitle(
    f"Variance of Source Channel Loading (topomap) — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_var[-1], label="variance")
fig_var.tight_layout()
if SAVE_PLOTS:
    fig_var.savefig(
        PLOTS_DIR / "ica_topomap_variance.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (e) — Per-Subject Mixing-Weight Bar Plot for Each Component

For each ICA component, compute the **mean absolute mixing weight**
across frequencies per subject.  This scalar summarises how strongly
each participant’s spectral profile participates in the mode.

### How this differs from subject-freq features (Approach 8)

In Approach 8, bars show mean absolute ICA **score** over frequencies.
Here, bars show mean absolute **mixing weight** over frequencies.
Both measure per-subject participation, but from opposite sides
of the decomposition.


In [ ]:
# Subject mixing: mean |weight| over frequencies
subject_mixing = np.abs(mixing_2d).mean(axis=1)  # (S, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_mixing[:, i],
        color="mediumpurple",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|mixing|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(
    f"Per-Subject Mixing Weight per Component — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_subject_mixing.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Summary

### Decomposition overview

This notebook uses the reshape `(C × T, S × F)` — channels and time
as observations, subjects and frequencies as features.  ICA discovers
**subject–spectral modes** — recurring subject × frequency patterns
that appear independently across channel–time observations.

| Aspect | Subject-Freq Features (Approach 8) | Inverted (here, Approach 9) |
|--------|------------------------------------|---------------------------------|
| Observations | S × F | C × T |
| Features | C × T | S × F |
| Components represent | Spatial-temporal modes `(C, T)` | (not used directly) |
| Sources represent | (not used directly) | Spatial-temporal activations `(C, T)` |
| Scores represent | Subject-frequency loadings `(S, F)` | (not used directly) |
| Mixing represents | (not used directly) | Subject-frequency mixing `(S, F)` |
| ISC based on | Score frequency vectors | Mixing frequency vectors |
| Temporal profiles from | Component channel-avg | Source channel-avg |
| Channel loadings from | Component time-avg | Source time-avg |
| Subject weights from | Mean |score| over F | Mean |mixing| over F |
